# Convert NAFNet checkpoint (.pth) sang ONNX

Notebook nay chay tren **Google Colab**, dung de:
1. Mount Google Drive (noi ban da luu file `.pth`)
2. Clone source code NAFNet + cai dependencies
3. Build lai kien truc model dung config da train, load checkpoint
4. Export sang ONNX (fixed input size, phu hop cho mobile)
5. Simplify + verify output ONNX khop voi PyTorch

Day la pipeline "san xuat" — chi lam dung 1 viec la convert. De kiem tra chat
luong ket qua sau convert (test anh that, so sanh voi ban goc...), dung file
rieng **`test_onnx_quality.ipynb`** — no tu load lai file `.onnx` da luu o Drive,
khong can chay lai notebook nay moi lan muon test.

> Truoc khi chay: **Runtime > Change runtime type > GPU** (khong bat buoc, chi giup load model nhanh hon).
> Sau khi sua CONFIG o Buoc 3, dung **Runtime > Run all** de chay toan bo tu dau.

## Buoc 0. Kiem tra GPU (tuy chon)

In [ ]:
!nvidia-smi

## Buoc 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Buoc 2. Lay source code NAFNet + cai dependencies

Clone full (khong dung `--depth 1`) va KHONG chay `python setup.py develop`
(hay bi loi `KeyError: __version__` khi thieu git history) — chi can them
thu muc repo vao `sys.path` la du de import `basicsr`.

In [ ]:
%cd /content
!rm -rf NAFNet
!git clone https://github.com/megvii-research/NAFNet.git
%cd /content/NAFNet

!pip install -q -r requirements.txt
!pip install -q onnx onnxruntime onnxsim onnxscript

In [ ]:
import sys
sys.path.insert(0, '/content/NAFNet')

import os
import torch
import numpy as np
from basicsr.models.archs.NAFNet_arch import NAFNet

print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## Buoc 3. CONFIG — chinh cac bien o day

- `CHECKPOINT_PATH`: duong dan toi file `.pth` tren Drive.
- `PRESET`: **phai khop chinh xac** voi checkpoint, lay truc tiep tu
  `options/train/*/NAFNet-*.yml` cua repo goc:
  - `sidd_width32` / `sidd_width64` — checkpoint SIDD (denoise): `enc=[2,2,4,8] mid=12 dec=[2,2,2,2]`
  - `gopro_width32` / `gopro_width64` — checkpoint GoPro (deblur): `enc=[1,1,1,28] mid=1 dec=[1,1,1,1]`
  - `reds_width64` — checkpoint REDS: giong GoPro, chi khac width=64
- `IMG_H`, `IMG_W`: kich thuoc tile co dinh khi export (boi so cua 16). Day cung
  la kich thuoc anh app mobile se phai crop/tile truoc khi dua vao model.

**Ghi lai dung 4 gia tri nay (CHECKPOINT_PATH, PRESET, IMG_H, IMG_W)** — file
`test_onnx_quality.ipynb` can nhap lai chinh xac cac gia tri nay de test dung.

Neu buoc 4 (load checkpoint) bao `size mismatch` -> ban dang chon sai `PRESET`
so voi `CHECKPOINT_PATH` — sua lai CA HAI dong cho khop nhau.

In [ ]:
CHECKPOINT_PATH = '/content/drive/MyDrive/Thesis/Product/Data/NAFNet-GoPro-width64.pth'  # <-- sua duong dan that
PRESET = 'gopro_width64'  # <-- phai khop voi CHECKPOINT_PATH o tren

PRESETS = {
    'sidd_width32':  dict(width=32, enc_blk_nums=[2, 2, 4, 8],  middle_blk_num=12, dec_blk_nums=[2, 2, 2, 2]),
    'sidd_width64':  dict(width=64, enc_blk_nums=[2, 2, 4, 8],  middle_blk_num=12, dec_blk_nums=[2, 2, 2, 2]),
    'gopro_width32': dict(width=32, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1,  dec_blk_nums=[1, 1, 1, 1]),
    'gopro_width64': dict(width=64, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1,  dec_blk_nums=[1, 1, 1, 1]),
    'reds_width64':  dict(width=64, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1,  dec_blk_nums=[1, 1, 1, 1]),
}
MODEL_CFG = PRESETS[PRESET]

IMG_H, IMG_W = 256, 256

ONNX_OUTPUT_PATH = f'/content/drive/MyDrive/NAFNet/nafnet_{PRESET}_{IMG_H}x{IMG_W}.onnx'
ONNX_SIMPLIFIED_PATH = ONNX_OUTPUT_PATH.replace('.onnx', '_sim.onnx')

size_mb = os.path.getsize(CHECKPOINT_PATH) / 1024 / 1024
print(f'Checkpoint: {CHECKPOINT_PATH} ({size_mb:.2f} MB)')
print('Preset:', PRESET, MODEL_CFG)
print('ONNX se luu tai:', ONNX_SIMPLIFIED_PATH)

## Buoc 4. Build model va load checkpoint

In [ ]:
model = NAFNet(img_channel=3, **MODEL_CFG)

ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
state_dict = ckpt.get('params', ckpt) if isinstance(ckpt, dict) else ckpt

missing, unexpected = model.load_state_dict(state_dict, strict=False)
assert len(missing) == 0 and len(unexpected) == 0, (
    f'State dict khong khop kien truc (missing={missing}, unexpected={unexpected}) '
    f'— kiem tra lai PRESET co dung voi CHECKPOINT_PATH khong.'
)
model.eval()

# Kiem tra nhanh trong so co "song" khong (phat hien checkpoint hong/rong)
for name in ['intro.weight', 'ending.weight']:
    p = dict(model.named_parameters())[name]
    print(f'{name}: mean={p.data.mean().item():.6f} std={p.data.std().item():.6f}')

print('Load checkpoint thanh cong.')

## Buoc 5. Export sang ONNX

In [ ]:
os.makedirs(os.path.dirname(ONNX_OUTPUT_PATH), exist_ok=True)

dummy_input = torch.randn(1, 3, IMG_H, IMG_W)

with torch.no_grad():
    torch.onnx.export(
        model,
        dummy_input,
        ONNX_OUTPUT_PATH,
        input_names=['input'],
        output_names=['output'],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,  # exporter kieu cu (TorchScript-based), on dinh hon voi kien truc custom
    )

print('Da export ONNX:', ONNX_OUTPUT_PATH)

## Buoc 6. Simplify ONNX graph

In [ ]:
import onnx
from onnxsim import simplify

# Dung Python API thay vi CLI 'python -m onnxsim' de tranh bug crash trong
# buoc in bang thong ke (sympy.factor treo voi so lon). Ket qua simplify khong doi.
model_onnx = onnx.load(ONNX_OUTPUT_PATH)
model_simplified, check = simplify(model_onnx)
assert check, 'Simplified ONNX model khong hop le'

onnx.save(model_simplified, ONNX_SIMPLIFIED_PATH)
print('Da simplify:', ONNX_SIMPLIFIED_PATH)

## Buoc 7. Verify: so sanh output ONNX vs PyTorch

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(ONNX_SIMPLIFIED_PATH, providers=['CPUExecutionProvider'])

test_input = torch.randn(1, 3, IMG_H, IMG_W)
with torch.no_grad():
    out_torch = model(test_input).numpy()
out_onnx = sess.run(None, {'input': test_input.numpy()})[0]

diff = np.abs(out_torch - out_onnx)
max_diff, mean_diff = diff.max(), diff.mean()
print('Max abs diff:', max_diff, '| Mean abs diff:', mean_diff)

# mean_diff la chi so chinh (dai dien cho toan anh). max_diff de canh bao neu
# qua bat thuong; model cang lon (width64) cang tich luy sai so lam tron nhieu hon.
assert mean_diff < 5e-3, 'Sai so TRUNG BINH qua lon — kiem tra lai export/simplify'
assert max_diff < 0.15, 'Sai so LON NHAT qua bat thuong — co the co loi export'
print('OK: output ONNX khop voi PyTorch.')

## Buoc 8. Hoan tat

File `.onnx` da duoc luu vao Google Drive tai:
- `ONNX_OUTPUT_PATH` (ban goc)
- `ONNX_SIMPLIFIED_PATH` (da simplify — nen dung file nay cho mobile)

**Buoc tiep theo:**
- Muon kiem tra chat luong ket qua (test anh that, so sanh voi ban PyTorch goc,
  test bang bo du lieu test chinh thuc...) -> mo file **`test_onnx_quality.ipynb`**,
  nhap lai dung `CHECKPOINT_PATH`, `PRESET`, `IMG_H`, `IMG_W`, `ONNX_SIMPLIFIED_PATH`
  o phan CONFIG cua file do.
- Muon tich hop vao app:
  - Android: dung truc tiep file `.onnx` voi ONNX Runtime Mobile (`onnxruntime-android`).
  - iOS: dung ONNX Runtime Mobile cho iOS, hoac convert tiep sang CoreML bang `coremltools`.
  - TFLite: dung `onnx2tf` hoac `onnx-tf` de convert ONNX -> TFLite.